In [165]:
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.workflow.pipeline import Pipeline

pipeline_session = PipelineSession()
region = pipeline_session.boto_region_name
role = sagemaker.get_execution_role()
bucket = pipeline_session.default_bucket()
prefix = "emi-y-mora/pipeline-byoc"

preprocess_image_uri = "780191826160.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-preprocess:latest"
train_image_uri = "780191826160.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-train:latest"
serve_image_uri = "780191826160.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-serve:finalv2"

processing_instance_type = ParameterString(
    name="ProcessingInstanceType",
    default_value="ml.m5.large"
)

training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.large"
)

transform_instance_type = ParameterString(
    name="TransformInstanceType",
    default_value="ml.m5.large"
)

model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval"
)

rmse_threshold = ParameterFloat(
    name="RMSEThreshold",
    default_value=1.0
)

print("role:", role)
print("bucket:", bucket)
print("region:", region)
print("prefix:", prefix)

role: arn:aws:iam::780191826160:role/SageMakerStudioExecutionRole2026
bucket: sagemaker-us-east-1-780191826160
region: us-east-1
prefix: emi-y-mora/pipeline-byoc


In [164]:
pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:780191826160:pipeline/EmiYMoraPipelineBYOC',
 'ResponseMetadata': {'RequestId': '77288fdd-f405-4adf-b128-cc353ca38916',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '77288fdd-f405-4adf-b128-cc353ca38916',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '110',
   'date': 'Thu, 26 Mar 2026 01:37:39 GMT'},
  'RetryAttempts': 0}}

In [166]:
from pathlib import Path

from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput, TransformInput
from sagemaker.model import Model
from sagemaker.transformer import Transformer
from sagemaker.model_metrics import MetricsSource, ModelMetrics

from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.fail_step import FailStep

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("repo_root:", repo_root)

repo_root: /home/sagemaker-user/Emi-y-Mora


In [167]:
preprocess_processor = ScriptProcessor(
    image_uri=preprocess_image_uri,
    command=["python"],
    role=role,
    instance_count=1,
    instance_type=processing_instance_type,
    sagemaker_session=pipeline_session,
    base_job_name="emi-y-mora-preprocess",
)

train_estimator = Estimator(
    image_uri=train_image_uri,
    role=role,
    instance_count=1,
    instance_type=training_instance_type,
    output_path=f"s3://{bucket}/{prefix}/training-output",
    sagemaker_session=pipeline_session,
    base_job_name="emi-y-mora-train",
)

In [168]:
step_process = ProcessingStep(
    name="Preprocess",
    processor=preprocess_processor,
    inputs=[
        ProcessingInput(
            source=str(repo_root / "data" / "raw"),
            destination="/opt/ml/processing/input",
            input_name="raw",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/output/validation",
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test",
        ),
    ],
    code=str(repo_root / "src" / "prep.py"),
)

In [169]:
step_train = TrainingStep(
    name="Train",
    estimator=train_estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    },
)

In [170]:
evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

step_evaluate = ProcessingStep(
    name="Evaluate",
    processor=preprocess_processor,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/input/model",
            input_name="model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/input/test",
            input_name="test",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/output/evaluation",
        )
    ],
    code=str(repo_root / "src" / "evaluate.py"),
    property_files=[evaluation_report],
)

In [171]:
serve_model = Model(
    image_uri=serve_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session,
    name="emi-y-mora-serve-model",
)

step_create_model = ModelStep(
    name="CreateModel",
    step_args=serve_model.create(
        instance_type=transform_instance_type,
    ),
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [172]:
transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_count=1,
    instance_type=transform_instance_type,
    output_path=f"s3://{bucket}/{prefix}/batch-output",
    sagemaker_session=pipeline_session,
)

step_transform = TransformStep(
    name="BatchTransform",
    transformer=transformer,
    inputs=TransformInput(
        data=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
        content_type="text/csv",  
    ),
)

In [173]:
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_evaluate.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
                "evaluation.json",
            ],
        ),
        content_type="application/json",
    )
)

step_register = RegisterModel(
    name="RegisterModel",
    estimator=train_estimator,
    image_uri=serve_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    model_package_group_name="EmiYMoraModelPackageGroup",
    approval_status=model_approval_status,
    content_types=["text/csv", "application/json"],
    response_types=["application/json"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_metrics=model_metrics,
)

In [174]:
rmse_value = JsonGet(
    step_name=step_evaluate.name,
    property_file=evaluation_report,
    json_path="regression_metrics.rmse.value",
)

step_fail = FailStep(
    name="FailOnHighRMSE",
    error_message=Join(
        on="",
        values=[
            "RMSE above threshold. Threshold=",
            rmse_threshold,
        ],
    ),
)

step_condition = ConditionStep(
    name="CheckRMSE",
    conditions=[
        ConditionLessThanOrEqualTo(
            left=rmse_value,
            right=rmse_threshold,
        )
    ],
    if_steps=[step_create_model, step_transform, step_register],
    else_steps=[step_fail],
)

In [175]:
pipeline = Pipeline(
    name="EmiYMoraPipelineBYOC",
    parameters=[
        processing_instance_type,
        training_instance_type,
        transform_instance_type,
        model_approval_status,
        rmse_threshold,
    ],
    steps=[
        step_process,
        step_train,
        step_evaluate,
        step_condition,
    ],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=role)
execution = pipeline.start()
print("Pipeline execution ARN:", execution.arn)

Pipeline execution ARN: arn:aws:sagemaker:us-east-1:780191826160:pipeline/EmiYMoraPipelineBYOC/execution/1zsqhdogepgi


In [176]:
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:780191826160:pipeline/EmiYMoraPipelineBYOC',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:780191826160:pipeline/EmiYMoraPipelineBYOC/execution/1zsqhdogepgi',
 'PipelineExecutionDisplayName': 'execution-1774489079718',
 'PipelineExecutionStatus': 'Executing',
 'PipelineExperimentConfig': {'ExperimentName': 'EmiYMoraPipelineBYOC',
  'TrialName': '1zsqhdogepgi'},
 'CreationTime': datetime.datetime(2026, 3, 26, 1, 37, 59, 647000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 3, 26, 1, 37, 59, 647000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:780191826160:user-profile/d-5k4j2nbjajkd/datascientist',
  'UserProfileName': 'datascientist',
  'DomainId': 'd-5k4j2nbjajkd',
  'IamIdentity': {'Arn': 'arn:aws:sts::780191826160:assumed-role/SageMakerStudioExecutionRole2026/SageMaker',
   'PrincipalId': 'AROA3LJYOQDYIZIAG3ONL:SageMaker'}},
 'LastModifiedBy': {'UserProfileArn': 'arn:aws:sagema

In [177]:
execution.list_steps()

[{'StepName': 'Train',
  'StartTime': datetime.datetime(2026, 3, 26, 1, 43, 4, 535000, tzinfo=tzlocal()),
  'StepStatus': 'Executing',
  'Metadata': {'TrainingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:780191826160:training-job/pipelines-1zsqhdogepgi-Train-es4VBMkOUc'}},
  'AttemptCount': 1},
 {'StepName': 'Preprocess',
  'StartTime': datetime.datetime(2026, 3, 26, 1, 38, 0, 825000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 3, 26, 1, 43, 3, 727000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'ProcessingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:780191826160:processing-job/pipelines-1zsqhdogepgi-Preprocess-UMBLmwg0aC'}},
  'AttemptCount': 1}]

In [181]:
steps = execution.list_steps()
for s in steps:
    print("STEP:", s["StepName"])
    print("  STATUS:", s["StepStatus"])
    print("  START :", s.get("StartTime"))
    print("  END   :", s.get("EndTime"))
    print("  FAIL  :", s.get("FailureReason"))
    print("-" * 60)

STEP: BatchTransform
  STATUS: Failed
  START : 2026-03-26 02:05:46.235000+00:00
  END   : 2026-03-26 02:40:21.281000+00:00
  FAIL  : ClientError: AlgorithmError: Model container failed to respond to ping. Please ensure /ping endpoint is implemented and responds with an HTTP 200 status code
------------------------------------------------------------
STEP: CreateModel-CreateModel
  STATUS: Succeeded
  START : 2026-03-26 02:05:43.263000+00:00
  END   : 2026-03-26 02:05:45.360000+00:00
  FAIL  : None
------------------------------------------------------------
STEP: RegisterModel-RegisterModel
  STATUS: Succeeded
  START : 2026-03-26 02:05:43.263000+00:00
  END   : 2026-03-26 02:05:45.175000+00:00
  FAIL  : None
------------------------------------------------------------
STEP: CheckRMSE
  STATUS: Succeeded
  START : 2026-03-26 02:05:42.235000+00:00
  END   : 2026-03-26 02:05:42.730000+00:00
  FAIL  : None
------------------------------------------------------------
STEP: Evaluate
  STAT

In [182]:
steps = execution.list_steps()
bt = [s for s in steps if s["StepName"] == "BatchTransform"][0]
job_arn = bt["Metadata"]["TransformJob"]["Arn"]
job_name = job_arn.split("/")[-1]
print(job_name)

pipelines-1zsqhdogepgi-BatchTransform-LKkvra1lXb


In [183]:
import boto3

logs = boto3.client("logs")
group = "/aws/sagemaker/TransformJobs"

streams = logs.describe_log_streams(
    logGroupName=group,
    logStreamNamePrefix=job_name,
    orderBy="LogStreamName",
    descending=True,
)

stream_name = streams["logStreams"][0]["logStreamName"]

events = logs.get_log_events(
    logGroupName=group,
    logStreamName=stream_name,
    startFromHead=True,
)

for e in events["events"][-120:]:
    print(e["message"])

[2026-03-26 02:10:13 +0000] [7] [INFO] Starting gunicorn 25.2.0
[2026-03-26 02:10:13 +0000] [7] [INFO] Listening at: http://0.0.0.0:8080 (7)
[2026-03-26 02:10:13 +0000] [7] [INFO] Using worker: sync
[2026-03-26 02:10:13 +0000] [8] [INFO] Booting worker with pid: 8
[2026-03-26 02:10:13 +0000] [8] [ERROR] Exception in worker process
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/gunicorn/arbiter.py", line 713, in spawn_worker
    worker.init_process()
  File "/usr/local/lib/python3.12/site-packages/gunicorn/workers/base.py", line 136, in init_process
    self.load_wsgi()
  File "/usr/local/lib/python3.12/site-packages/gunicorn/workers/base.py", line 148, in load_wsgi
    self.wsgi = self.app.wsgi()
                ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/gunicorn/app/base.py", line 66, in wsgi
    self.callable = self.load()
                    ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/gunicorn/app/wsgiapp.py", 